# Medical Insurance Cost Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `charges`

## 2 · Project Overview

We predict **medical insurance charges** based on policyholder demographics (age, sex), health indicators (BMI, smoking status), family size (children), and geographic region.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a policyholder's age, sex, BMI, number of children, smoking status, and region, predict their annual medical insurance charges.

## 5 · Why This Project Matters

- **Insurance pricing** is a core actuarial problem.
- Understanding cost drivers supports underwriting decisions.
- The smoking × BMI interaction reveals non-linear cost structure.
- Demonstrates regression with a strong categorical driver (smoker).

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,000 |
| **Features** | age, sex, bmi, children, smoker, region |
| **Target** | charges (continuous, USD) |
| **Range** | ~$1K – $65K |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "charges"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: charges
Save dir: E:\Github\Machine-Learning-Projects\Regression\Medical Insurance Cost Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 7,000 insurance policyholders with demographic and health features.

In [4]:
np.random.seed(SEED)
n = 7000
age = np.random.randint(18, 65, n)
sex = np.random.choice(["male", "female"], n)
bmi = np.round(np.random.normal(28, 6, n).clip(15, 55), 1)
children = np.random.poisson(1, n).clip(0, 5)
smoker = np.random.choice(["yes", "no"], n, p=[0.2, 0.8])
region = np.random.choice(["northeast", "southeast", "southwest", "northwest"], n)

smoker_mult = np.where(smoker == "yes", 4.5, 1.0)
charges = (250 * age + 300 * bmi + 500 * children + 1000) * smoker_mult
charges = np.round(charges + np.random.normal(0, 1500, n), 2).clip(1000, 65000)

df = pd.DataFrame({"age": age, "sex": sex, "bmi": bmi, "children": children,
                    "smoker": smoker, "region": region, "charges": charges})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['charges'].describe()}")
df.head()

Dataset shape: (7000, 7)
Target stats:
count     7000.000000
mean     29356.452770
std      18371.688722
min       7601.770000
25%      17895.970000
50%      21815.020000
75%      26689.280000
max      65000.000000
Name: charges, dtype: float64


,age,sex,bmi,children,smoker,region,charges
0,56,male,31.0,0,no,southwest,25313.94
1,46,male,35.6,3,no,northwest,26480.54
2,32,male,26.9,1,no,southwest,17559.90
3,60,male,31.9,2,no,northwest,25686.95
4,25,male,24.8,2,no,northeast,13268.71


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

DATA VALIDATION

Shape: (7000, 7)

Missing values:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Duplicate rows: 4

Dtypes:
age           int32
sex          object
bmi         float64
children      int32
smoker       object
region       object
charges     float64
dtype: object

Target 'charges' confirmed.

Target stats:
count     7000.000000
mean     29356.452770
std      18371.688722
min       7601.770000
25%      17895.970000
50%      21815.020000
75%      26689.280000
max      65000.000000
Name: charges, dtype: float64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["age"], df[TARGET], alpha=0.2, s=10, c=df["smoker"].map({"yes":"red","no":"blue"}))
axes[0][0].set_xlabel("Age"); axes[0][0].set_ylabel("Charges"); axes[0][0].set_title("Age vs Charges (red=smoker)")

axes[0][1].scatter(df["bmi"], df[TARGET], alpha=0.2, s=10, c=df["smoker"].map({"yes":"red","no":"blue"}))
axes[0][1].set_xlabel("BMI"); axes[0][1].set_ylabel("Charges"); axes[0][1].set_title("BMI vs Charges (red=smoker)")

df.boxplot(column=TARGET, by="smoker", ax=axes[0][2])
axes[0][2].set_title("Charges by Smoking Status")

df.boxplot(column=TARGET, by="region", ax=axes[1][0])
axes[1][0].set_title("Charges by Region")

df.boxplot(column=TARGET, by="children", ax=axes[1][1])
axes[1][1].set_title("Charges by # Children")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=axes[1][2], square=True)
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `charges`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Charges ($)")

axes[1].hist(df[df["smoker"]=="yes"][TARGET], bins=20, alpha=0.7, label="Smoker", color="red", edgecolor="black")
axes[1].hist(df[df["smoker"]=="no"][TARGET], bins=20, alpha=0.7, label="Non-smoker", color="blue", edgecolor="black")
axes[1].set_title("Charges by Smoking Status")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")
print(f"Smoker mean: ${df[df['smoker']=='yes'][TARGET].mean():,.0f}")
print(f"Non-smoker mean: ${df[df['smoker']=='no'][TARGET].mean():,.0f}")

Range: $7,602 to $65,000
Mean: $29,356, Median: $21,815
Smoker mean: $64,634
Non-smoker mean: $20,196


## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

Train: (5600, 6), Test: (1400, 6)
Target — Train mean: 29,520.92, Test mean: 28,698.57


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `sex`, `smoker`, `region` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Smokers form a distinct high-cost cluster — keep them.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["age_bmi"] = X_train["age"] * X_train["bmi"]
X_test["age_bmi"] = X_test["age"] * X_test["bmi"]

X_train["bmi_over30"] = (X_train["bmi"] > 30).astype(int)
X_test["bmi_over30"] = (X_test["bmi"] > 30).astype(int)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (8): ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'age_bmi', 'bmi_over30']


## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

BASELINE: Linear Regression
RMSE : 1,927.23
MAE  : 1,510.71
R2   : 0.9885


## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Regressors:
                               Adjusted R-Squared  R-Squared         RMSE  Time Taken
Model                                                                                
GradientBoostingRegressor                0.993832   0.993868  1408.640100    0.460715
HistGradientBoostingRegressor            0.993546   0.993583  1441.011900    0.254161
LGBMRegressor                            0.993462   0.993499  1450.362121    0.111141
CatBoostRegressor                        0.993427   0.993465  1454.213579    2.164384
RandomForestRegressor                    0.993064   0.993104  1493.807321    1.220267
XGBRegressor                             0.992931   0.992972  1508.064105    3.242971
BaggingRegressor                         0.992755   0.992796  1526.736330    0.147806
ExtraTreesRegressor                      0.992431   0.992475  1560.457512    0.916033
AdaBoostRegressor                        0.992403   0.992447  1563.366401    0.195551
KNeighborsRegressor  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

FLAML best model: lgbm
RMSE: 1,440.03
R2  : 0.9936


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost RMSE: 1,415.34  (1.4s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[111]	valid_0's l2: 2.02521e+06
LightGBM RMSE: 1,423.10  (1.2s)


XGBoost failed: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by RMSE):
                      RMSE      MAE      R2
CatBoost           1415.34  1036.87  0.9938
LightGBM           1423.10  1037.25  0.9937
FLAML              1440.03  1038.75  0.9936
Linear Regression  1927.23  1510.71  0.9885

Top 3 models: ['CatBoost', 'LightGBM', 'FLAML']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")


Detailed Metrics — Top 3:

  CatBoost:
    RMSE : 1,415.34
    MAE  : 1,036.87
    R2   : 0.9938

  LightGBM:
    RMSE : 1,423.10
    MAE  : 1,037.25
    R2   : 0.9937

  FLAML:
    RMSE : 1,440.03
    MAE  : 1,038.75
    R2   : 0.9936


## 24 · Error Analysis

Examine residuals from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

Residual stats (CatBoost):
  Mean: 59.85, Std: 1,414.07
  Min: -5,221.44, Max: 6,338.18
  Median: 16.22


## 25 · Interpretation and Business Insight

**Key findings:**
- **Smoking status** is the single strongest cost driver (~4.5× multiplier).
- **Age** has a strong linear effect on charges.
- **BMI** interacts with smoking — obese smokers pay dramatically more.
- **Children** add moderate incremental cost.
- **Region** and **sex** have minimal independent effect.

**Business takeaway:** Smoking is the dominant rating factor. BMI amplifies costs primarily among smokers.

## 26 · Limitations

1. Synthetic data with a simple multiplicative model.
2. No pre-existing conditions or medical history.
3. Missing plan type (HMO, PPO, deductible).
4. No claims frequency/severity breakdown.
5. Region effect is unrealistically uniform.

## 27 · How to Improve This Project

1. Use real insurance claims data.
2. Add pre-existing conditions and prescription history.
3. Include plan type and deductible level.
4. Model claims frequency and severity separately.
5. Add time dimension for cost trend analysis.

## 28 · Production Considerations

- Deploy for automated premium calculation.
- Output prediction intervals for risk tiers.
- Monitor claims vs. predictions quarterly.
- Ensure regulatory compliance (no prohibited rating factors).
- Calibrate predictions to loss ratios.

## 29 · Common Mistakes

1. Not recognizing the bimodal distribution (smoker vs. non-smoker).
2. Using linear regression without interaction terms.
3. Not log-transforming right-skewed charges.
4. Ignoring the smoker × BMI interaction.
5. Using prohibited rating factors (race, genetics).

## 30 · Mini Challenge / Exercises

1. Log-transform `charges` and compare Linear Regression R².
2. Add `smoker × bmi` interaction feature manually.
3. Build separate models for smokers and non-smokers.
4. Remove `region` — does RMSE change?
5. Plot residuals by smoking status — are they homoscedastic?

## 31 · Final Summary / Key Takeaways

1. **Smoking** is the dominant cost driver (~4.5× multiplier).
2. **Age** and **BMI** provide strong linear signals.
3. **Smoker × BMI** interaction captures the highest-cost subgroup.
4. **Region** and **sex** have minimal predictive power.
5. **Tree models** naturally capture the smoking interaction.
6. **Actuarial models** separate frequency from severity.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Regression\Medical Insurance Cost Prediction\metrics.json
{
  "CatBoost": {
    "rmse": 1415.34,
    "mae": 1036.87,
    "r2": 0.9938
  },
  "LightGBM": {
    "rmse": 1423.1,
    "mae": 1037.25,
    "r2": 0.9937
  },
  "Linear Regression": {
    "rmse": 1927.23,
    "mae": 1510.71,
    "r2": 0.9885
  },
  "FLAML": {
    "rmse": 1440.03,
    "mae": 1038.75,
    "r2": 0.9936
  }
}
